## 1. 라이브러리 및 환경 설정

In [38]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold

## 2. 데이터 로드

In [39]:
layout = pd.read_csv('./data/layout_info.csv')

if 'layout_type' not in train.columns:
    train = train.merge(layout, on='layout_id', how='left')
    test = test.merge(layout, on='layout_id', how='left')

print(f"학습 데이터 크기: {train.shape}")
print(f"테스트 데이터 크기: {test.shape}")

학습 데이터 크기: (250000, 134)
테스트 데이터 크기: (50000, 133)


In [48]:
print([col for col in train.columns if 'layout_type' in col])
print([col for col in test.columns if 'layout_type' in col])

# ===== 파생 변수 생성 =====
train['order_per_robot'] = train['order_inflow_15m'] / (train['robot_active'] + 1)
test['order_per_robot'] = test['order_inflow_15m'] / (test['robot_active'] + 1)

train['charge_pressure'] = train['robot_charging'] / (train['charger_count'] + 1)
test['charge_pressure'] = test['robot_charging'] / (test['charger_count'] + 1)

train['congestion_per_robot'] = train['congestion_score'] / (train['robot_active'] + 1)
test['congestion_per_robot'] = test['congestion_score'] / (test['robot_active'] + 1)

# train/test 동일 기준으로 layout_type 인코딩
combined = pd.concat([train['layout_type'], test['layout_type']], axis=0).astype(str)
mapping = {v: i for i, v in enumerate(combined.unique())}

train['layout_type'] = train['layout_type'].astype(str).map(mapping)
test['layout_type'] = test['layout_type'].astype(str).map(mapping)

print(train['layout_type'].head())
print(train['layout_type'].dtype)

['layout_type']
['layout_type']
0    0
1    0
2    0
3    0
4    0
Name: layout_type, dtype: int64
int64


## 3. 피처 및 타겟 설정

In [51]:
TARGET = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']

feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
print(f"피처 수: {len(feature_cols)}")

피처 수: 133


## 4. 모델 학습 (GroupKFold)

In [52]:
X = train[feature_cols]
y = train[TARGET]
groups = train['scenario_id']

kf = GroupKFold(n_splits=5)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y, groups=groups)):
    print(f"── Fold {fold + 1} ──")

    X_tr = train.loc[train_idx, feature_cols]
    y_tr = train.loc[train_idx, TARGET]
    X_val = train.loc[valid_idx, feature_cols]
    y_val = train.loc[valid_idx, TARGET]

    model = LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=7,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=42,
        verbose=-1,
    )

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
    )

    oof_preds[valid_idx] = model.predict(X_val)
    test_preds += model.predict(test[feature_cols]) / 5

oof_preds = np.clip(oof_preds, 0, None)
test_preds = np.clip(test_preds, 0, None)

── Fold 1 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 529.61
Early stopping, best iteration is:
[81]	valid_0's l2: 528.367
── Fold 2 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 497.834
Early stopping, best iteration is:
[80]	valid_0's l2: 497.411
── Fold 3 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 502.322
Early stopping, best iteration is:
[119]	valid_0's l2: 501.715
── Fold 4 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 383.627
[200]	valid_0's l2: 383.453
Early stopping, best iteration is:
[155]	valid_0's l2: 382.306
── Fold 5 ──
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 456.209
Early stopping, best iteration is:
[134]	valid_0's l2: 455.025


## 5. 결과 확인

In [53]:
oof_mae = mean_absolute_error(train[TARGET], oof_preds)
print(f"OOF MAE: {oof_mae:.4f}")

OOF MAE: 9.8407


## 6. 제출 파일 생성

In [54]:
submission = pd.DataFrame({'ID': test['ID'], TARGET: test_preds})
submission.to_csv('./submission.csv', index=False)
print("submission.csv 저장 완료.")

submission.csv 저장 완료.
